# 🐎 Hermes Agent — Colab + Google Drive + Telegram (v3)

Ye notebook [Hermes Agent](https://github.com/NousResearch/hermes-agent) (NousResearch) ko Google Colab par chalata hai:

- Config, `.env`, skills aur sessions **Google Drive** ke `harmes agent` folder me persist hote hain
- Pehli baar 3 cheezein poochega: **Telegram Bot Token**, **Telegram User ID**, **OpenRouter API key**
- **Live OpenRouter free-model list** dikhata hai — aap specific model choose kar sakte hain, ya Enter dabakar auto (recommended) choose ho jayega
- **Automatic fallback**: agar chuna hua model fail ho (rate-limit, credits khatam, provider error) to **khud-ba-khud free model par switch** ho jayega — bina conversation kate
- Telegram par tool-progress ek hi message **edit-in-place karke live update** dikhata hai

## Pehli baar use karne se pehle:
1. Telegram par [@BotFather](https://t.me/BotFather) ko `/newbot` bhejkar bot banayein → **token copy karein**
2. [@userinfobot](https://t.me/userinfobot) ko koi message bhejein → apni **numeric User ID** note karein
3. [openrouter.ai/keys](https://openrouter.ai/keys) se free account bana kar **API key** le lein

## Use karne ka tareeka:
`Runtime → Run all`. Credential wali cell me 3 sawaal + model choice poochega. Agli baar Drive se sab restore ho jayega.

⚠️ Colab free tier sessions kuch ghanton baad disconnect ho sakti hain — testing/personal bot ke liye theek hai.

## 1) Google Drive Mount Karein

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = "/content/drive/MyDrive/harmes agent"
BACKUP_DIR = f"{DRIVE_BASE}/hermes_data"
os.makedirs(BACKUP_DIR, exist_ok=True)
print("Drive folder ready:", BACKUP_DIR)


## 2) Hermes Agent Install Karein

Code/venv Colab ki **local disk** par install hota hai (Drive FUSE par venv chalana bahut slow hota hai) — har naye session me ~2-5 minute mein dobara install hoga, ye normal hai.

In [ ]:
import subprocess, os, shutil

def sh(cmd, cwd=None):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)

def find_bin(name):
    return sh(f"command -v {name}").stdout.strip()

def already_installed():
    return os.path.exists(os.path.expanduser("~/.hermes/hermes-agent")) or os.path.exists("/usr/local/lib/hermes-agent")

sh("apt-get -qq update && apt-get -qq install -y curl xz-utils git > /dev/null 2>&1")

if not already_installed():
    print("Installing Hermes Agent (pehli baar is session me, thoda time lagega)...")

    # Colab par uv pehle se /usr/local/bin par hota hai jo Hermes ke installer ko confuse
    # kar deta hai (usse ~/.hermes/bin me uv expect karta hai) — pehle hi copy kar dete hain.
    uv_path = find_bin("uv")
    if uv_path:
        os.makedirs(os.path.expanduser("~/.hermes/bin"), exist_ok=True)
        shutil.copy2(uv_path, os.path.expanduser("~/.hermes/bin/uv"))
        uvx_path = find_bin("uvx")
        if uvx_path:
            shutil.copy2(uvx_path, os.path.expanduser("~/.hermes/bin/uvx"))

    result = sh("curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash -s -- --skip-browser")

    if result.returncode != 0 and "uv installer reported success but binary not found" in result.stdout:
        print("uv mismatch phir se detect hua, retry kar rahe hain...")
        uv_path = find_bin("uv")
        if uv_path:
            os.makedirs(os.path.expanduser("~/.hermes/bin"), exist_ok=True)
            shutil.copy2(uv_path, os.path.expanduser("~/.hermes/bin/uv"))
        result = sh("curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash -s -- --skip-browser")

    print(result.stdout[-2500:])
    if result.returncode != 0:
        print("----STDERR----")
        print(result.stderr[-1500:])
        print("\n⚠️ Install fail hui — upar output check karein.")
    else:
        print("\n✅ Hermes install complete.")
else:
    print("Hermes is session me pehle se installed hai, install skip.")

for c in [os.path.expanduser("~/.local/bin"), "/usr/local/bin"]:
    if os.path.exists(os.path.join(c, "hermes")) and c not in os.environ.get("PATH", ""):
        os.environ["PATH"] = c + ":" + os.environ["PATH"]

INSTALL_DIR = "/usr/local/lib/hermes-agent" if os.path.isdir("/usr/local/lib/hermes-agent") else os.path.expanduser("~/.hermes/hermes-agent")
VENV_PYTHON = os.path.join(INSTALL_DIR, "venv", "bin", "python")
print("Install dir:", INSTALL_DIR)
print("Venv python milta hai:", os.path.exists(VENV_PYTHON))

check = sh("hermes --version")
print(check.stdout.strip() or check.stderr.strip())


## 3) Telegram Support Install Karein

Base installer `python-telegram-bot` install nahi karta — seedha venv me daal dete hain (idempotent).

In [ ]:
check = subprocess.run(f'"{VENV_PYTHON}" -c "import telegram"', shell=True, capture_output=True, text=True)

if check.returncode != 0:
    print("python-telegram-bot install kar rahe hain...")
    r = sh(f'uv pip install --python "{VENV_PYTHON}" "python-telegram-bot"', cwd=INSTALL_DIR)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("----STDERR----")
        print(r.stderr[-1000:])
else:
    print("python-telegram-bot already installed, skip.")

verify = subprocess.run(f'"{VENV_PYTHON}" -c "import telegram; print(telegram.__version__)"', shell=True, capture_output=True, text=True)
print("python-telegram-bot version:", verify.stdout.strip() or verify.stderr.strip())


## 4) Pichla Saved Config Drive Se Restore Karein

In [ ]:
import shutil

HERMES_HOME = os.path.expanduser("~/.hermes")
os.makedirs(HERMES_HOME, exist_ok=True)

files_to_sync = [".env", "config.yaml"]
dirs_to_sync = ["skills", "sessions"]

def restore_from_drive():
    restored = False
    for f in files_to_sync:
        src = f"{BACKUP_DIR}/{f}"
        if os.path.exists(src):
            shutil.copy2(src, f"{HERMES_HOME}/{f}")
            restored = True
    for d in dirs_to_sync:
        src = f"{BACKUP_DIR}/{d}"
        if os.path.exists(src):
            os.makedirs(f"{HERMES_HOME}/{d}", exist_ok=True)
            sh(f'rsync -a "{src}/" "{HERMES_HOME}/{d}/"')
            restored = True
    return restored

had_previous_setup = restore_from_drive()
print("Pichla config Drive se restore ho gaya ✅" if had_previous_setup else "Koi purana config nahi mila — niche pehli-baar wala setup chalega 👇")


## 5) Model Chunein (OpenRouter Free Models)

Live list OpenRouter se fetch karta hai — sirf **free** (₹0) models, jinka context ≥64K ho (Hermes ki minimum requirement). `0` dabakar **Auto** (recommended) bhi chun sakte hain — wo hamesha koi available free model use karega, kabhi delist nahi hota.

In [ ]:
import requests

def pick_model():
    print("OpenRouter ke available FREE models fetch kar rahe hain...\n")
    try:
        resp = requests.get("https://openrouter.ai/api/v1/models", timeout=15)
        all_models = resp.json().get("data", [])
    except Exception as e:
        print("Model list fetch nahi ho payi (", e, "), auto-free use karenge.")
        return "openrouter/openrouter/free"

    free_models = []
    for m in all_models:
        pricing = m.get("pricing", {})
        try:
            is_free = float(pricing.get("prompt", "1")) == 0 and float(pricing.get("completion", "1")) == 0
        except (ValueError, TypeError):
            is_free = False
        if is_free and m.get("context_length", 0) >= 64000:
            free_models.append(m)

    free_models.sort(key=lambda m: m.get("context_length", 0), reverse=True)
    free_models = free_models[:15]

    print("0) Auto (recommended) \u2014 hamesha koi available free model, kabhi delist nahi hota")
    for i, m in enumerate(free_models, start=1):
        ctx = m.get("context_length", 0)
        model_id = m.get("id", "")
        print(f"{i}) {model_id}  (context: {ctx:,})")

    choice = input(f"\nModel chunein [0-{len(free_models)}], Enter = 0 (auto): ").strip()

    if choice.isdigit() and 1 <= int(choice) <= len(free_models):
        picked_id = free_models[int(choice)-1].get("id", "")
        return f"openrouter/{picked_id}"
    return "openrouter/openrouter/free"

def ensure_fallback_to_free():
    """Agar primary model fail ho (rate-limit/credits/error) to free par auto-switch."""
    config_path = os.path.expanduser("~/.hermes/config.yaml")
    config_text = open(config_path).read() if os.path.exists(config_path) else ""
    if "fallback_providers:" not in config_text:
        with open(config_path, "a") as f:
            f.write("\nfallback_providers:\n  - provider: openrouter\n    model: openrouter/free\n")
        print("\u2705 Fallback set: primary model fail hone par khud openrouter/free par switch hoga")
    else:
        print("Fallback pehle se configured hai, skip.")


## 6) Pehli Baar Ka Setup (Telegram Token + OpenRouter Key)

Sirf tab poochega jab pehli baar chal rahe ho. Secrets `.env` me seedha Python se likhte hain (`hermes config set` nahi) taaki formatting kharab na ho.

In [ ]:
env_path = os.path.expanduser("~/.hermes/.env")
existing_env = open(env_path).read() if os.path.exists(env_path) else ""

def env_set(key, value, current_text):
    lines = [l for l in current_text.splitlines() if l.strip() and not l.startswith(f"{key}=")]
    lines.append(f"{key}={value}")
    return "\n".join(lines) + "\n"

if "TELEGRAM_BOT_TOKEN" not in existing_env or "OPENROUTER_API_KEY" not in existing_env:
    print("=== Pehli baar ka setup (sirf ek hi baar karna hoga) ===\n")

    tg_token = input("Telegram Bot Token (@BotFather se): ").strip()
    tg_user_id = input("Aapki Telegram numeric User ID (@userinfobot se): ").strip()
    or_key = input("OpenRouter API key (openrouter.ai/keys se, sk-or- se shuru hogi): ").strip()

    text = existing_env
    text = env_set("TELEGRAM_BOT_TOKEN", tg_token, text)
    text = env_set("TELEGRAM_ALLOWED_USERS", tg_user_id, text)
    text = env_set("OPENROUTER_API_KEY", or_key, text)
    with open(env_path, "w") as f:
        f.write(text)

    model_value = pick_model()
    sh(f"hermes config set model {model_value}")
    sh("hermes config set platforms.telegram.enabled true")
    sh("hermes config set display.platforms.telegram.tool_progress new")
    ensure_fallback_to_free()

    print("\n✅ Setup ho gaya! Model:", model_value)
else:
    print("Config Drive se pehle hi restore ho chuka hai — credential step skip.")
    ensure_fallback_to_free()   # purane restored config me bhi fallback zaroor ho


## 6b) [Optional] Model Kabhi Bhi Badlein

Ye cell sirf tab chalayein jab aap **model badalna chahte ho** (gateway band karke). Baaki sab kuch waisa hi rehta hai.

In [ ]:
# Sirf tab chalayein jab model badalna ho:
new_model = pick_model()
sh(f"hermes config set model {new_model}")
print("Model update ho gaya:", new_model)


## 7) Pre-flight Check

In [ ]:
print("=== Pre-flight check ===\n")

env_text = open(env_path).read() if os.path.exists(env_path) else ""
env_dict = dict(l.split("=",1) for l in env_text.splitlines() if "=" in l)
config_text = open(os.path.expanduser("~/.hermes/config.yaml")).read() if os.path.exists(os.path.expanduser("~/.hermes/config.yaml")) else ""

checks = {
    "TELEGRAM_BOT_TOKEN .env me hai": len(env_dict.get("TELEGRAM_BOT_TOKEN","")) > 5,
    "TELEGRAM_ALLOWED_USERS .env me hai": len(env_dict.get("TELEGRAM_ALLOWED_USERS","")) > 0,
    "OPENROUTER_API_KEY .env me hai": len(env_dict.get("OPENROUTER_API_KEY","")) > 5,
    "platforms.telegram.enabled = true": "true" in sh("hermes config get platforms.telegram.enabled").stdout,
    "model.default set hai": "openrouter" in sh("hermes config get model").stdout,
    "fallback_providers configured hai": "fallback_providers:" in config_text,
    "python-telegram-bot installed hai": subprocess.run(f'"{VENV_PYTHON}" -c "import telegram"', shell=True, capture_output=True).returncode == 0,
}

all_ok = True
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
    all_ok = all_ok and ok

print("\n" + ("🚀 Sab theek hai — ab backup aur gateway cell chalayein." if all_ok else "⚠️ Kuch missing hai upar — usse pehle fix karein."))


## 8) Config Ko Drive Par Backup Karein

In [ ]:
def backup_to_drive():
    sh(f'cp -f ~/.hermes/.env "{BACKUP_DIR}/.env" 2>/dev/null')
    sh(f'cp -f ~/.hermes/config.yaml "{BACKUP_DIR}/config.yaml" 2>/dev/null')
    sh(f'mkdir -p "{BACKUP_DIR}/skills" "{BACKUP_DIR}/sessions"')
    sh(f'rsync -a ~/.hermes/skills/ "{BACKUP_DIR}/skills/" 2>/dev/null')
    sh(f'rsync -a ~/.hermes/sessions/ "{BACKUP_DIR}/sessions/" 2>/dev/null')
    print("Drive par backup ho gaya ✅")

backup_to_drive()


## 9) Telegram Gateway Chalu Karein 🚀

Chalne ke baad apne Telegram bot ko koi bhi message bhej sakte hain. Rokne ke liye is cell ka Stop/Interrupt (⏹) button dabayein — har 5 minute me aur rukne par bhi Drive par auto-backup hota hai.

In [ ]:
import threading, time

def auto_backup_loop():
    while True:
        time.sleep(300)
        backup_to_drive()

threading.Thread(target=auto_backup_loop, daemon=True).start()

print("🚀 Hermes gateway chalu ho raha hai... Telegram par apne bot ko message bhejein.")
print("Rokne ke liye: is cell ka Stop/Interrupt (⏹) button dabayein.\n")

process = subprocess.Popen(
    ["hermes", "gateway"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
try:
    for line in process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    pass
finally:
    process.terminate()
    backup_to_drive()
    print("\nGateway ruk gaya — latest state Drive par backup ho gaya.")
